# Run a cluster notebook for every cluster in Caminha et al. (2021)

This notebook runs one cluster notebook (by default `cluster_cosmo_models.ipynb`)
for each cluster marked in the `Caminha` column of
`example_data/venus_clusters_summary.csv`, one after the other, with rj2129
first. Folder names come from the `glafic_name` column.

Each cluster runs in a fresh kernel with `GLAFIC_CLUSTER` set, just as when you
run the notebook by hand. The executed notebook is saved with that cluster's
other results, as `results/<cluster>/<notebook>.executed.ipynb`
(`results_smoke/` when `SMOKE = True`).

If a cluster fails (for example, the results-folder guard stops it or the
model fails the image check), the failure is logged and that cluster is
skipped; the others still run. Re-running is cheap for finished clusters:
every stage loads its cached result and the samplers resume from their
checkpoints. Progress is written to `results/run_caminha.log`.

This is the same as `run_caminha.py`. For batch jobs (e.g. `srun` on NERSC),
use the script; it doesn't need a running Jupyter kernel.

## Settings

In [ ]:
NOTEBOOK = "cluster_cosmo_models.ipynb"   # notebook in this folder to run (or "cluster_cosmo_hmc.ipynb")
CLUSTERS = None                           # None: Caminha's clusters; or a list of folder names, e.g. ["rj2129", "mj1931"]
SMOKE = False                             # True: tiny chains (SMOKE_TEST = True), results in results_smoke/
DRY_RUN = False                           # True: only list the clusters
FIRST = "rj2129"                          # run this cluster first

In [ ]:
import csv
import os
import re
import sys
import time
import traceback

import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

HERE = os.path.abspath(os.getcwd())      # run this notebook from examples_jax/cluster_cosmo
if not os.path.exists(os.path.join(HERE, "build_cluster.py")):
    raise RuntimeError(f"run this notebook from examples_jax/cluster_cosmo (cwd is {HERE})")
sys.path.insert(0, HERE)
import build_cluster as build


def caminha_clusters():
    # folder names of the clusters marked in the summary table's 'Caminha' column, FIRST first
    path = os.path.join(build.DATA, build.SUMMARY)
    rows = list(csv.DictReader(open(path)))
    names = [r["glafic_name"].strip() for r in rows if (r.get("Caminha") or "").strip()]
    if not names:
        raise RuntimeError(f"no rows marked in the 'Caminha' column of {path}")
    return sorted(names, key=lambda n: (n != FIRST, names.index(n)))


def log(msg, logfile):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(logfile, "a") as f:
        f.write(line + "\n")


async def run_one(cluster, notebook, results, smoke, logfile):
    # execute the notebook for one cluster in a fresh kernel; returns (ok, seconds)
    nb = nbformat.read(os.path.join(HERE, notebook), as_version=4)
    if smoke:
        n = 0
        for c in nb.cells:
            if c.cell_type == "code" and "SMOKE_TEST = False" in c.source:
                c.source = c.source.replace("SMOKE_TEST = False", "SMOKE_TEST = True"); n += 1
        if n != 1:
            raise RuntimeError(f"{notebook}: could not switch on SMOKE_TEST")
    outdir = os.path.join(HERE, results, cluster)
    os.makedirs(outdir, exist_ok=True)
    out = os.path.join(outdir, notebook.replace(".ipynb", ".executed.ipynb"))
    os.environ["GLAFIC_CLUSTER"] = cluster              # the new kernel inherits it
    client = NotebookClient(nb, timeout=None, kernel_name="python3",
                            resources={"metadata": {"path": HERE}})
    t0 = time.time()
    try:
        await client.async_execute()                    # async: this kernel's event loop is already running
        ok = True
    except CellExecutionError as err:
        ok = False
        plain = re.sub(r"\x1b\[[0-9;]*m", "", str(err))   # drop the traceback's terminal colours
        last = [ln for ln in plain.strip().splitlines() if ln.strip()]
        log(f"{cluster}: FAILED in a notebook cell: {last[-1] if last else err}", logfile)
    finally:
        nbformat.write(nb, out)                         # keep the outputs, also of a failed run
    log(f"{cluster}: {'done' if ok else 'stopped'} in {(time.time() - t0) / 60:.1f} min; "
        f"executed notebook: {os.path.relpath(out, HERE)}", logfile)
    return ok, time.time() - t0

## Clusters to run

In [ ]:
if not os.path.exists(os.path.join(HERE, NOTEBOOK)):
    raise FileNotFoundError(f"no notebook {NOTEBOOK} in {HERE} (run its make_notebook*.py first)")
clusters = list(CLUSTERS or caminha_clusters())
unknown = [c for c in clusters if c not in build.available()]
if unknown:
    raise ValueError(f"unknown clusters {unknown}; available: {build.available()}")
excluded = [c for c in clusters if build.is_excluded(c)]
results = "results_smoke" if SMOKE else "results"
logfile = os.path.join(HERE, results, "run_caminha.log")
os.makedirs(os.path.dirname(logfile), exist_ok=True)

print(f"{NOTEBOOK} for {len(clusters)} cluster(s): {', '.join(clusters)}" + (" (SMOKE_TEST)" if SMOKE else ""))
if excluded:
    print(f"note: {excluded} are marked 'exclude'; the notebook stops for them unless RUN_EXCLUDED = True")

## Run

Each cluster's own output goes to its executed notebook, not here; this cell
only prints progress. Interrupting the kernel stops the run; the next run
picks up from the cached stages and checkpoints.

In [ ]:
status = {}
if DRY_RUN:
    print("DRY_RUN = True: nothing run")
else:
    log(f"start: {NOTEBOOK} for {', '.join(clusters)}" + (" (SMOKE_TEST)" if SMOKE else ""), logfile)
    for c in clusters:
        log(f"{c}: starting", logfile)
        try:
            status[c] = await run_one(c, NOTEBOOK, results, SMOKE, logfile)
        except Exception:                               # kernel died, etc.: log and go on
            log(f"{c}: ERROR outside the notebook:\n{traceback.format_exc()}", logfile)
            status[c] = (False, 0.0)
    log("summary: " + ", ".join(f"{c} {'ok' if ok else 'FAILED'} ({t / 60:.0f} min)"
                                for c, (ok, t) in status.items()), logfile)

In [ ]:
if status:
    print(f"{'cluster':12s} {'status':8s} {'minutes':>8s}  executed notebook")
    for c, (ok, t) in status.items():
        nb_out = os.path.join(results, c, NOTEBOOK.replace(".ipynb", ".executed.ipynb"))
        print(f"{c:12s} {'ok' if ok else 'FAILED':8s} {t / 60:8.1f}  {nb_out}")